# **Final Model Saving and Deployment Preparation**


This notebook prepares the selected final machine learning model for deployment in the coronary heart disease prediction system. The tuned XGBoost pipeline is selected as the deployment model because it demonstrated strong predictive performance, achieved high sensitivity for identifying positive coronary heart disease cases, and supported detailed interpretation through SHAP and LIME analyses.

The deployment preparation process involves loading the selected model, saving it under a clear deployment-ready filename, preserving the expected feature order, recording relevant model metadata, and verifying that all required files are available. These steps ensure that the model can be integrated consistently into an application programming interface or clinical decision-support interface without altering the preprocessing and prediction workflow established during model development.

## **Library Imports**

The required Python libraries are imported to support model loading, file management, metadata creation, and data validation. These libraries enable the final model and its associated deployment artefacts to be saved in structured formats for subsequent application integration.

In [1]:
# Data handling
import pandas as pd

# Model persistence
import joblib

# Metadata storage
import json

# File and directory management
from pathlib import Path

# Date and time recording
from datetime import datetime

## **Model Ingestion**

The tuned XGBoost pipeline selected for deployment is loaded from the model directory. Because the saved object contains the fitted resampling and classification components used during model development, the complete pipeline is retained to preserve consistency between training and deployment. The loaded object is then inspected to confirm that the correct model has been retrieved successfully.

In [2]:
# Define the path to the selected deployment model
selected_model_path = Path(
    "../models/xgboost_tuned.pkl"
)

# Load the tuned XGBoost pipeline
final_model = joblib.load(
    selected_model_path
)

print("Selected deployment model loaded successfully.\n")

print(f"Model file: {selected_model_path.name}")
print(f"Model object type: {type(final_model).__name__}")

# Inspect the pipeline steps
if hasattr(final_model, "named_steps"):
    print("\nPipeline steps:")
    
    for step_name, step_object in final_model.named_steps.items():
        print(
            f"- {step_name}: "
            f"{type(step_object).__name__}"
        )
else:
    print("\nThe loaded object is not a pipeline.")

Selected deployment model loaded successfully.

Model file: xgboost_tuned.pkl
Model object type: Pipeline

Pipeline steps:
- resampler: SMOTEENN
- model: XGBClassifier


### **Saving the Final Deployment Model**

The selected deployment model is saved using a descriptive filename to distinguish it from the intermediate models generated during model development. Assigning a standardized filename simplifies model deployment and ensures that downstream applications consistently reference the correct production-ready model.

In [3]:
# Define the deployment model path
deployment_model_path = Path(
    "../models/final_chd_model.pkl"
)

# Save the deployment model
joblib.dump(
    final_model,
    deployment_model_path
)

print("Deployment model saved successfully.\n")

print(f"Saved model path: {deployment_model_path}")
print(f"File exists: {deployment_model_path.exists()}")

Deployment model saved successfully.

Saved model path: ..\models\final_chd_model.pkl
File exists: True


### **Saving Feature Names**

The feature names used during model training are saved to preserve the expected input structure of the deployment model. Storing the feature order ensures that future prediction requests provide the predictor variables in the correct sequence, thereby preventing input mismatches and improving the robustness of the deployed coronary heart disease prediction system.

In [8]:
# Load the training feature dataset
X_train = pd.read_csv(
    "../data/processed/X_train.csv"
)

# Extract the feature names
feature_names = X_train.columns.tolist()

# Define the output path
feature_names_path = Path(
    "../models/feature_names.json"
)

# Save the feature names
with open(feature_names_path, "w") as file:
    json.dump(
        feature_names,
        file,
        indent=4
    )

print("Feature names saved successfully.\n")

print(f"Number of features: {len(feature_names)}")

print("\nFeature order:")

for index, feature in enumerate(feature_names, start=1):
    print(f"{index}. {feature}")

Feature names saved successfully.

Number of features: 15

Feature order:
1. male
2. age
3. education
4. currentSmoker
5. cigsPerDay
6. BPMeds
7. prevalentStroke
8. prevalentHyp
9. diabetes
10. totChol
11. sysBP
12. diaBP
13. BMI
14. heartRate
15. glucose


### **Saving Model Metadata**

Model metadata are recorded to document key information about the deployed machine learning model. The metadata include the deployment model name, learning algorithm, prediction task, target variable, number of predictor variables, feature names, and the date of creation. Recording these details promotes reproducibility, facilitates model management, and provides useful documentation for future deployment, maintenance, and dissertation reporting.

In [6]:
# Create the model metadata
model_metadata = {
    "model_name": "Final Coronary Heart Disease Prediction Model",
    "file_name": "final_chd_model.pkl",
    "algorithm": "Extreme Gradient Boosting (XGBoost)",
    "model_type": "Binary Classification",
    "target_variable": "TenYearCHD",
    "number_of_features": len(feature_names),
    "feature_names": feature_names,
    "pipeline_components": list(final_model.named_steps.keys()),
    "metrics": {
        "accuracy": 0.672,
        "precision": 0.254,
        "recall": 0.597,
        "f1_score": 0.356,
        "roc_auc": 0.667
    }
}

In [9]:
# Define the metadata path
metadata_path = Path(
    "../models/model_metadata.json"
)

# Save the metadata
with open(metadata_path, "w") as file:
    json.dump(
        model_metadata,
        file,
        indent=4
    )

print("Model metadata saved successfully.\n")

print("Metadata Summary:\n")

for key, value in model_metadata.items():
    if key == "metrics":
        print(f"{key}:")
        for metric_name, metric_value in value.items():
            print(f"  - {metric_name}: {metric_value}")
    else:
        print(f"{key}: {value}")

Model metadata saved successfully.

Metadata Summary:

model_name: Final Coronary Heart Disease Prediction Model
file_name: final_chd_model.pkl
algorithm: Extreme Gradient Boosting (XGBoost)
model_type: Binary Classification
target_variable: TenYearCHD
number_of_features: 15
feature_names: ['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
pipeline_components: ['resampler', 'model']
metrics:
  - accuracy: 0.672
  - precision: 0.254
  - recall: 0.597
  - f1_score: 0.356
  - roc_auc: 0.667


### **Verification of Saved Deployment Files**

The saved deployment artefacts are reloaded to verify that they have been stored correctly and are ready for deployment. This verification step confirms that the deployment model, feature names, and model metadata can be successfully retrieved without errors, thereby ensuring the integrity of the saved files.

In [10]:
# Reload the saved deployment artifacts
deployment_model = joblib.load(
    "../models/final_chd_model.pkl"
)

with open("../models/feature_names.json", "r") as file:
    deployment_feature_names = json.load(file)

with open("../models/model_metadata.json", "r") as file:
    deployment_metadata = json.load(file)

print("Deployment artifacts verified successfully.\n")

print(f"Deployment model: {type(deployment_model).__name__}")
print(f"Number of deployment features: {len(deployment_feature_names)}")
print(f"Deployment algorithm: {deployment_metadata['algorithm']}")

Deployment artifacts verified successfully.

Deployment model: Pipeline
Number of deployment features: 15
Deployment algorithm: Extreme Gradient Boosting (XGBoost)


## **Overall Deployment Summary**

The deployment preparation process successfully generated all artefacts required for integrating the developed coronary heart disease prediction model into a production environment. The tuned XGBoost pipeline was selected as the final deployment model and saved using a standardized filename to facilitate consistent model loading within external applications.

In addition to the deployment model, the feature names were preserved to ensure that future prediction requests follow the same predictor ordering used during model development. Model metadata were also recorded to document the characteristics and predictive performance of the deployed model, thereby supporting reproducibility, deployment management, and future maintenance.

Finally, all deployment artefacts were successfully reloaded and verified, confirming that the model, feature names, and metadata were stored correctly and are ready for deployment in a clinical decision-support application or web-based prediction system.